## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/YzbY9vJ.png)


This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

 ✅ Answer:

The three states in this Deep Research system are hierarchically organized to support a **supervisor-researcher delegation architecture**:

### The Three States:

1. **AgentState (Top Level - Lines 65-72)**
   - Contains: `messages`, `supervisor_messages`, `research_brief`, `notes`, `raw_notes`, `final_report`
   - Scope: The entire research workflow from start to finish

2. **SupervisorState (Middle Level - Lines 74-81)**
   - Contains: `supervisor_messages`, `research_brief`, `notes`, `raw_notes`, `research_iterations`
   - Scope: The supervisor's delegation and coordination activities

3. **ResearcherState (Bottom Level - Lines 83-90)**
   - Contains: `researcher_messages`, `tool_call_iterations`, `research_topic`, `compressed_research`, `raw_notes`
   - Scope: Individual researcher's focused research task

### How They Interrelate:

**Data Flow (Top → Down):**
- AgentState passes `research_brief` to SupervisorState
- SupervisorState spawns multiple ResearcherStates, each with a specific `research_topic`

**Data Flow (Bottom → Up):**
- Each ResearcherState produces `compressed_research` and `raw_notes`
- These are aggregated into SupervisorState's `notes` and `raw_notes`
- SupervisorState's accumulated findings flow back to AgentState
- AgentState uses all `notes` to generate the `final_report`

### Why Not a Single Huge State?

**1. Isolation & Parallelization**
- Multiple researchers run in parallel, each with their own isolated state
- If they shared one state, parallel execution would cause race conditions and conflicts
- Separate ResearcherStates allow 5+ researchers to work simultaneously without interference

**2. Scope Management**
- Each level only sees data relevant to its responsibility
- Researchers don't need to know about the overall conversation or other researchers' work
- Supervisor doesn't need individual researcher's message history, only their findings
- This prevents context pollution and keeps prompts focused

**3. Memory Efficiency**
- Each researcher's full message history (with tool calls) stays in ResearcherState
- Only compressed findings bubble up to higher levels
- Without separation, the top-level state would accumulate all tool calls from all researchers, quickly exceeding token limits

**4. Iteration Control**
- Each level has its own iteration counter (`research_iterations`, `tool_call_iterations`)
- Allows independent control: supervisor can delegate 6 times, each researcher can make 10 tool calls
- With one state, these limits would interfere with each other

**5. Clean Subgraph Boundaries**
- Each state maps to a subgraph with clear inputs/outputs
- ResearcherState → researcher_subgraph (input: research_topic, output: compressed_research)
- SupervisorState → supervisor_subgraph (spawns researchers, aggregates results)
- This modularity makes the system easier to understand, test, and modify

**The Trade-off:**
A single state would be simpler to understand initially, but would make parallel execution impossible, cause token limit issues, and create a tangled mess of responsibilities. The hierarchical state design is essential for this architecture to work effectively.

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

✅ Answer:

The notebook imports utilities, prompts, state definitions, and node functions from the `open_deep_library` package instead of defining them inline. Here's the analysis:

### Advantages of Importing from Library:

**1. Reusability & Modularity**
- The same components can be used across multiple notebooks, scripts, or applications
- Changes to core logic propagate automatically to all consumers
- Promotes DRY (Don't Repeat Yourself) principle

**2. Maintainability**
- Easier to update and debug - fix once in the library, benefits everywhere
- Clear separation of concerns: library handles implementation, notebook handles demonstration
- Version control is cleaner with organized file structure

**3. Readability & Focus**
- The notebook stays focused on **what** the system does, not **how** it works
- Reduces cognitive load - readers see the workflow without implementation details
- Makes the notebook a better teaching/documentation tool

**4. Testing & Quality**
- Library code can have proper unit tests, type hints, and documentation
- Easier to enforce code quality standards
- Can be packaged and distributed via PyPI

**5. Performance**
- Imported modules are compiled and cached by Python
- Notebook cells don't need to re-execute utility definitions
- Faster notebook execution and iteration

**6. Collaboration**
- Multiple developers can work on library code without notebook conflicts
- Clear API boundaries make it easier to divide work
- Professional development practices (linting, formatting) apply to library code

### Disadvantages of Importing from Library:

**1. Learning Curve**
- New users must navigate multiple files to understand the full system
- Harder to see the complete picture in one place
- Requires understanding of Python package structure

**2. Debugging Complexity**
- Stack traces span multiple files
- Can't easily modify and test changes in the notebook
- Need to restart kernel after library changes (or use `importlib.reload()`)

**3. Self-Contained Execution**
- Notebook isn't standalone - requires the library to be installed
- Sharing the notebook alone won't work without the package
- Dependency management becomes necessary

**4. Development Friction**
- Iterating on library code requires:
  - Edit library file → Save → Restart kernel → Re-run cells
- Slower feedback loop compared to notebook-only development
- Can't use notebook's interactive features for library code

**5. Discoverability**
- Important implementation details are "hidden" in library files
- Readers might not know where to look for specific functionality
- Documentation becomes critical to guide users

**6. Versioning Complexity**
- Notebook and library versions must stay synchronized
- Breaking changes in library can break notebooks
- Need proper semantic versioning and changelog

### The Trade-off:

**For Learning/Exploration:** Including code in the notebook is better - everything is visible and modifiable in one place.

**For Production/Distribution:** Importing from a library is better - promotes best practices, reusability, and maintainability.

**This Notebook's Approach:** It strikes a balance by:
- Importing stable, complex components (state, prompts, nodes)
- Keeping configuration and execution logic in the notebook
- Providing clear comments with line numbers pointing to library code
- Making it educational while maintaining professional structure

The choice depends on your goal: quick experimentation favors inline code, while building a robust system favors library imports.

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [7]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [8]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [9]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [10]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [11]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [12]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [13]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [14]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [15]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [16]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [17]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [18]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [19]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with your analysis request. You've provided a comprehensive PDF document (NBER Working Paper 34255: "How People Use ChatGPT") and requested insights on three key areas:

1. Main findings about how people are using AI
2. Most common use cases
3. Trends and patterns emerging from the data

The document contains detailed research data on ChatGPT usage patterns from November 2022 through July 2025, including usage classifications, demographic trends, and work vs. non-work applications. I will now begin analyzing this research paper to provide comprehensive insights addressing your three focus areas.

Node: write_research_brief

Research Brief Generated:
I have a PDF document from the National Bureau of Economic Research (NBER Working Paper 34255: "How People Use ChatGPT" by Chatterji et al., September 2025) that analyzes ChatGPT usage patterns from November 2022 through July 2025. I need you t


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# How People Use ChatGPT: Comprehensive Analysis of NBER Working Paper 34255

## Main Findings About AI Usage Patterns

### Unprecedented Global Adoption Scale
The NBER study reveals that ChatGPT has achieved extraordinary adoption rates, reaching over 700 million weekly active users by July 2025—representing approximately 10% of the global adult population [1][2][3]. This makes it the fastest-growing consumer application in history, reaching 1 million users in just 5 days and 100 million users in less than 2 months [4][5]. The platform now processes over 2.6 billion messages per day, equivalent to more than 30,000 messages per second [6].

### Dramatic Demographic Evolution
One of the most striking findings is the complete closure of the gender gap in ChatGPT usage. Initially, over 80% of users had typically male names when the platform first launched in November 2022. However, by July 2025, 52% of active users had typically female names, indicating that the gender gap has not only narrowed but has been completely eliminated [6][7]. This shift occurred gradually, with the percentage of users with typically feminine names rising from 37% in January 2024 to over half by July 2025 [8].

### Age Distribution and User Composition
The research reveals that ChatGPT has a young user base, with nearly half of all messages coming from users under 26 years old [9][10]. The detailed age breakdown shows that users aged 25-34 represent the largest group at 33.52%, followed by 18-24 year-olds at 28.04%. Older demographics have lower representation, with users 65+ accounting for only 3.33% of the user base [4].

### Global Geographic Expansion
The study demonstrates ChatGPT's worldwide reach, with availability in 188 countries and support for 59 languages [10][11]. The United States leads adoption with 14.82% of global traffic, followed by India at 8.18%. Importantly, the research found higher growth rates in lower-income countries, with usage now similar between countries at different GDP levels—Brazil, South Korea, and the US have comparable usage despite vastly different per capita incomes [6].

## Most Common Use Cases and Message Classifications

### The Big Three: Dominant Usage Categories
The research identifies that nearly 80% of all ChatGPT conversations fall into three primary categories [4][8][12]:

**Practical Guidance (29%)**: This is the most common use case, encompassing activities like tutoring and teaching, how-to advice across various topics, and creative ideation. Over one-third of practical guidance messages focus specifically on tutoring or teaching, making education a major component of ChatGPT usage [13].

**Seeking Information (24%)**: This category includes searching for information about people, current events, products, and recipes. The research notes that this appears to be a very close substitute for traditional web search engines, suggesting ChatGPT is competing directly with Google and other search platforms [8][12].

**Writing (24%)**: This category encompasses the automated production of emails, documents, and other communications, but also includes editing, critiquing, summarizing, and translating text provided by users. Notably, about two-thirds of writing requests involve editing or improving existing text rather than creating documents from scratch [13].

### Work vs. Non-Work Usage Patterns
A critical finding is the dramatic shift in work-related versus non-work usage over time. Non-work messages have grown substantially from 53% in mid-2024 to over 70% in mid-2025 [4][8][12][14]. Work-related usage shows steady but slower growth, now accounting for approximately 27-30% of all conversations. This shift primarily reflects changing usage patterns within existing user cohorts rather than changes in the composition of new users [8].

### Message Classification by Interaction Type
The study categorized messages into three interaction types based on user intent [4][14][15]:
- **Asking**: ~49% (seeking advice, information, guidance)
- **Doing**: ~40% (task execution like writing, coding)
- **Expressing**: ~11% (casual chat, expressing views/feelings)

Importantly, "Asking" interactions are growing faster than "Doing" activities and receive higher user satisfaction ratings, suggesting users find advisory functions more valuable than pure task automation [15].

### Professional Usage Characteristics
For work-related usage, the patterns differ significantly from personal use. Work usage heavily skews toward "Doing" activities at approximately 56% [4]. Writing dominates work-related tasks, accounting for approximately 40% of all work messages [12][13]. The research found that work usage correlates strongly with education levels and high-paying professional occupations, with educated users and those in professional roles being more likely to use ChatGPT for work purposes [8][14].

### Programming Usage Reality vs. Perception
Contrary to popular assumptions about ChatGPT being primarily a coding tool, computer programming represents only about 4.2% of all messages [4][12][13]. This finding challenges common perceptions about how the platform is used and suggests that non-programming applications dominate actual usage patterns.

## Trends and Patterns Emerging from the Data

### Explosive Growth Trajectories
The research reveals remarkable growth patterns across multiple metrics. Weekly active users have doubled every 7-8 months since launch, while message volume has grown 5.8 times in the past year—faster than user growth, indicating more intensive usage over time [6]. The platform has evolved from processing millions of messages to billions, with specific milestones including:
- December 2022: 1 million users
- November 2023: 100 million weekly users
- December 2024: 300 million weekly users
- February 2025: 400 million weekly users
- July 2025: 700+ million weekly users [4][8]

### Cohort Behavior and Product Improvement
A fascinating finding is that all user signup cohorts follow similar usage trajectories—relatively flat through most of 2024, then substantial increases beginning in late 2024/early 2025 [6]. This pattern suggests that ChatGPT has become significantly better and more user-friendly over time, rather than reflecting users gradually learning how to use the platform more effectively.

### Shift Toward Personal and Consumer Applications
The dramatic increase in non-work usage from 53% to over 70% represents a fundamental shift in how the platform is being used. This trend suggests that while much economic analysis of AI has focused on productivity impacts in paid work, the impact on personal activities and home production may be equally or more significant [8][12]. This finding aligns with research by Collis and Brynjolfsson (2025) estimating consumer surplus of at least $97 billion in 2024 alone in the US.

### User Engagement and Satisfaction Trends
User satisfaction remains consistently high, with positive interactions outnumbering negative ones by approximately 4:1 [4]. Users spend an average of 12 minutes and 9 seconds per session, viewing an average of 4.5 pages per visit [8]. The highest satisfaction ratings come from advisory interactions like tutoring and guidance rather than task completion, suggesting users derive more value from decision support than from pure automation [13].

### Enterprise and Commercial Adoption
The research documents significant enterprise adoption, with ChatGPT serving 1.5 million enterprise customers and being used by 92% of Fortune 500 companies [11]. Professional adoption varies by role:
- Software developers: 79%
- Journalists: 64%
- Marketing professionals: 65%
- IT support specialists: 55%
- HR experts: 45% [10][13]

### Revenue and Subscription Trends
The platform has achieved remarkable commercial success with over 10 million paying subscribers across Plus, Team, and Pro plans [8][11]. Subscriber retention is exceptionally strong, with 89% of users continuing their subscription after one quarter and 74% remaining subscribed after three quarters [11]. OpenAI generated $3.7 billion in revenue in 2024 and reached $10 billion ARR by June 2025 [11].

### Market Position and Competitive Landscape
ChatGPT has established dominant market positions across multiple categories:
- 62.5% of the AI assistant market [11]
- 79.76% of the chatbot market [11]
- 5th most visited website globally with over 5.2 billion monthly visits [8][10]

### Economic Value Creation and Implications
The research concludes that ChatGPT's primary economic value lies in decision support rather than simple task automation. The platform's strongest contribution is as a decision-support system that helps people weigh options, choose better words, and interpret information [13]. This finding has significant implications for understanding AI's economic impact, suggesting the technology's value extends beyond productivity gains in traditional work tasks to encompass broader cognitive support across all aspects of life.

The democratization of access across demographic groups and countries, combined with the shift toward personal and consumer applications, suggests that AI's societal impact may be broader and more evenly distributed than initially anticipated. The research indicates that ChatGPT provides economic value through improved decision-making capabilities, which is particularly important in knowledge-intensive jobs but extends to personal decisions and learning as well.

### Sources

[1] TECHMANIACS.com: How People Really Use ChatGPT: Findings from NBER Research
[2] SSRN: How People Use ChatGPT
[3] Forked Lightning: How People Use ChatGPT - by David Deming
[4] LinkedIn: ChatGPT usage: 700M weekly users, 73% non-work, 80% practical
[5] NBER: How People Use ChatGPT | NBER
[6] LinkedIn: OpenAI releases research on ChatGPT usage worldwide
[7] LinkedIn: ChatGPT usage: 80% of conversations fall into three categories
[8] Exploding Topics: Number of ChatGPT Users (October 2025)


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs